# Bagian A — Linear Regression

## A1. Deskripsi Dataset

Dataset yang dipakai adalah **California Housing Prices** dari Kaggle. Sumbernya adalah [California Housing Prices](https://www.kaggle.com/datasets/camnugent/california-housing-prices). Data ini berisi informasi rumah dan wilayah di California, Amerika Serikat.

Satu baris mewakili satu wilayah. Dataset ini memiliki sekitar **20.640 baris dan 10 kolom**. Delapan kolom berisi informasi yang dipakai sebagai fitur, satu kolom berisi kategori lokasi, dan satu kolom adalah target.

Arti kolomnya sebagai berikut:

- `longitude`: posisi wilayah dari arah barat ke timur.
- `latitude`: posisi wilayah dari arah selatan ke utara.
- `housing_median_age`: usia tengah rumah di wilayah tersebut.
- `total_rooms`: jumlah seluruh ruangan di wilayah tersebut.
- `total_bedrooms`: jumlah seluruh kamar tidur di wilayah tersebut.
- `population`: jumlah penduduk di wilayah tersebut.
- `households`: jumlah rumah tangga di wilayah tersebut.
- `median_income`: pendapatan tengah penduduk. Angkanya memakai satuan puluhan ribu dolar.
- `ocean_proximity`: keterangan jarak wilayah dari laut. Kolom ini berupa teks, bukan angka.
- `median_house_value`: nilai tengah rumah di wilayah tersebut.

Target yang akan diprediksi adalah `median_house_value`. Kolom lain dipakai sebagai informasi untuk membantu model memperkirakan nilai rumah.

## Mengambil dan membaca dataset

Data diambil langsung dari Kaggle dengan `kagglehub`. Setelah itu, file CSV yang ada di folder hasil download dibaca dengan pandas.

In [ ]:
!pip -q install kagglehub

import os
import glob
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

path = kagglehub.dataset_download("camnugent/california-housing-prices")
csv_files = glob.glob(os.path.join(path, '*.csv'))
print('File yang ditemukan:', csv_files)
df = pd.read_csv(csv_files[0])
print('Ukuran data:', df.shape)
display(df.head())


## A2. Eksplorasi dan Visualisasi Data

Sebelum membuat model, kita lihat dulu bentuk datanya. Statistik deskriptif membantu melihat nilai rata-rata, nilai paling kecil, nilai paling besar, dan penyebaran data.

In [ ]:
print('Informasi kolom:')
df.info()
print('\nStatistik deskriptif:')
display(df.describe(include='all').T)
print('\nJumlah data kosong tiap kolom:')
print(df.isna().sum())
print('\nJumlah data duplikat:', df.duplicated().sum())


### Visualisasi 1: Histogram target

Histogram ini menunjukkan penyebaran `median_house_value`. Dari sini kita bisa melihat apakah nilai rumah banyak berkumpul di angka tertentu atau menyebar cukup lebar. Jika bentuknya tidak simetris, berarti ada bagian data yang lebih padat atau lebih jarang.

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df['median_house_value'], kde=True)
plt.title('Distribusi Median House Value')
plt.xlabel('Nilai tengah rumah')
plt.ylabel('Jumlah wilayah')
plt.show()


### Visualisasi 2: Heatmap korelasi

Heatmap digunakan untuk melihat hubungan antar kolom yang berbentuk angka. Angka korelasi mendekati 1 berarti hubungan positif, sedangkan angka mendekati -1 berarti hubungan negatif. Untuk pemilihan fitur, kita terutama melihat hubungan setiap fitur dengan `median_house_value`.

In [ ]:
numeric_df = df.select_dtypes(include=np.number)
plt.figure(figsize=(10, 7))
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Korelasi Antar Kolom Angka')
plt.show()


### Visualisasi 3: Scatter plot fitur terhadap target

Scatter plot berikut membandingkan `median_income` dengan `median_house_value`. Jika titik-titik cenderung naik dari kiri ke kanan, pendapatan yang lebih tinggi biasanya diikuti nilai rumah yang lebih tinggi. Titik yang menyebar jauh dari pola menunjukkan bahwa pendapatan bukan satu-satunya faktor penentu.

In [ ]:
sampel = df.sample(min(3000, len(df)), random_state=42)
plt.figure(figsize=(8, 5))
sns.scatterplot(data=sampel, x='median_income', y='median_house_value', alpha=.35)
plt.title('Median Income dan Median House Value')
plt.xlabel('Pendapatan tengah')
plt.ylabel('Nilai tengah rumah')
plt.show()


## A3. Preprocessing

Preprocessing adalah tahap menyiapkan data agar bisa dipakai oleh model.

- Data duplikat dihapus karena baris yang sama tidak perlu dihitung dua kali.
- Data kosong pada `total_bedrooms` diisi dengan median. Median dipilih karena lebih aman jika ada nilai yang terlalu besar atau terlalu kecil.
- `ocean_proximity` berisi kata seperti `NEAR BAY`, sehingga diubah menjadi kolom angka dengan one-hot encoding.
- Fitur angka distandardisasi. Tujuannya agar kolom dengan nilai besar tidak terlihat lebih penting hanya karena satuannya besar.
- Outlier dicek dengan metode IQR. Pada notebook ini, nilai yang terlalu jauh pada fitur angka dibatasi sampai batas IQR. Cara ini dipilih agar jumlah baris tidak berkurang terlalu banyak. Target tidak dipotong karena target adalah nilai yang memang ingin diprediksi.

Semua langkah yang dipelajari dari data train dimasukkan ke pipeline. Dengan begitu, data test tidak ikut dipakai saat model belajar.

In [ ]:
# Hapus baris yang sama persis
df = df.drop_duplicates().copy()

# Membatasi nilai ekstrem pada fitur angka dengan aturan IQR.
# Target tidak ikut dibatasi.
numeric_for_outlier = [
    'longitude', 'latitude', 'housing_median_age', 'total_rooms',
    'total_bedrooms', 'population', 'households', 'median_income'
]
for col in numeric_for_outlier:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    batas_bawah = q1 - 1.5 * iqr
    batas_atas = q3 + 1.5 * iqr
    df[col] = df[col].clip(batas_bawah, batas_atas)

X = df.drop(columns='median_house_value')
y = df['median_house_value']

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()
print('Fitur angka:', numeric_features)
print('Fitur kategori:', categorical_features)


## A4. Pemilihan Fitur

Tidak semua fitur harus dipakai dengan cara yang sama. Fitur angka dan fitur kategori diproses dulu, kemudian `SelectKBest` memilih 8 fitur yang paling berhubungan dengan target berdasarkan uji statistik `f_regression`.

Fitur yang terpilih dipertahankan oleh model. Fitur yang tidak terpilih tidak digunakan dalam perhitungan akhir. Nama fitur yang terpilih dan yang dibuang akan ditampilkan setelah model dilatih.

## A5. Pembagian Data

Data dibagi menjadi dua bagian:

- **Data latih 80%**: dipakai model untuk belajar.
- **Data uji 20%**: dipakai untuk mengecek hasil model pada data yang belum pernah dilihat.

`random_state=42` dipakai supaya pembagian data tetap sama setiap kali notebook dijalankan.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

numeric_process = Pipeline([
    ('isi_data_kosong', SimpleImputer(strategy='median')),
    ('scaling', StandardScaler())
])

categorical_process = Pipeline([
    ('isi_data_kosong', SimpleImputer(strategy='most_frequent')),
    ('encoding', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocess = ColumnTransformer([
    ('angka', numeric_process, numeric_features),
    ('kategori', categorical_process, categorical_features)
])

model = Pipeline([
    ('preprocess', preprocess),
    ('pilih_fitur', SelectKBest(f_regression, k=8)),
    ('regresi', LinearRegression())
])

model.fit(X_train, y_train)
print('Jumlah data latih:', len(X_train))
print('Jumlah data uji:', len(X_test))


## A6. Pemodelan

Model yang digunakan adalah `LinearRegression`. Model ini membuat rumus yang menghubungkan fitur dengan target.

- **Intercept** adalah nilai awal prediksi ketika pengaruh fitur dianggap nol.
- **Koefisien** menunjukkan arah pengaruh fitur. Koefisien positif berarti kenaikan fitur cenderung menaikkan prediksi. Koefisien negatif berarti kenaikan fitur cenderung menurunkan prediksi.

Karena fitur sudah melalui scaling, besar koefisien dipakai untuk melihat fitur mana yang lebih kuat pengaruhnya. Interpretasi ini tetap perlu hati-hati karena hubungan antar fitur bisa saling berkaitan.

In [ ]:
# Mendapatkan nama semua fitur setelah encoding
feature_names = model.named_steps['preprocess'].get_feature_names_out()
selected_mask = model.named_steps['pilih_fitur'].get_support()
selected_features = feature_names[selected_mask]
discarded_features = feature_names[~selected_mask]

reg = model.named_steps['regresi']
coef_table = pd.DataFrame({
    'fitur': selected_features,
    'koefisien': reg.coef_,
    'nilai_absolut': np.abs(reg.coef_)
}).sort_values('nilai_absolut', ascending=False)

print('Intercept:', reg.intercept_)
print('\nFitur yang dipertahankan:')
display(coef_table)
print('Fitur yang tidak dipakai:', list(discarded_features))


## A7. Evaluasi

Model dinilai dengan empat ukuran.

- **MAE** menunjukkan rata-rata selisih antara nilai asli dan prediksi. Semakin kecil semakin baik.
- **MSE** memberi hukuman lebih besar pada kesalahan yang besar. Semakin kecil semakin baik.
- **RMSE** adalah akar dari MSE, sehingga satuannya sama dengan target.
- **R-squared atau R2** menunjukkan seberapa besar perubahan target yang bisa dijelaskan oleh model. Nilai yang lebih dekat ke 1 biasanya lebih baik.

In [ ]:
pred_train = model.predict(X_train)
pred_test = model.predict(X_test)

def evaluasi(y_asli, y_prediksi, nama):
    hasil = {
        'MAE': mean_absolute_error(y_asli, y_prediksi),
        'MSE': mean_squared_error(y_asli, y_prediksi),
        'RMSE': np.sqrt(mean_squared_error(y_asli, y_prediksi)),
        'R-squared': r2_score(y_asli, y_prediksi)
    }
    print(nama)
    for nama_metrik, nilai in hasil.items():
        print(f'{nama_metrik}: {nilai:.4f}')
    print()
    return hasil

hasil_train = evaluasi(y_train, pred_train, 'Hasil Data Latih')
hasil_test = evaluasi(y_test, pred_test, 'Hasil Data Uji')


### Grafik aktual dan prediksi

Jika model bekerja dengan baik, titik-titik akan mendekati garis merah. Garis merah menunjukkan kondisi ketika nilai prediksi sama dengan nilai asli.

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(x=y_test, y=pred_test, alpha=.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Nilai asli')
plt.ylabel('Nilai prediksi')
plt.title('Nilai Aktual dan Nilai Prediksi')
plt.show()


### Residual plot

Residual adalah selisih antara nilai asli dan nilai prediksi. Jika titik-titik menyebar di sekitar garis nol tanpa pola tertentu, hasil model lebih masuk akal. Jika terlihat pola atau banyak titik jauh dari garis nol, berarti masih ada kesalahan yang belum bisa dijelaskan model.

In [ ]:
residual = y_test - pred_test
plt.figure(figsize=(8, 5))
sns.scatterplot(x=pred_test, y=residual, alpha=.5)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Nilai prediksi')
plt.ylabel('Residual')
plt.title('Residual Plot')
plt.show()


## A8. Analisis Performa

Bandingkan hasil data latih dan data uji. Jika R-squared data latih jauh lebih tinggi daripada data uji, sedangkan error data latih jauh lebih kecil, model bisa mengalami **overfitting**. Artinya model terlalu mengikuti data latih dan kurang baik saat bertemu data baru.

Jika hasil data latih dan data uji sama-sama rendah, model bisa mengalami **underfitting**. Artinya pola yang dipelajari belum cukup untuk menjelaskan data.

Fitur yang paling berpengaruh dapat dilihat dari nilai absolut koefisien terbesar pada tabel koefisien. Koefisien positif menunjukkan arah pengaruh yang naik, sedangkan koefisien negatif menunjukkan arah pengaruh yang turun. Karena fitur sudah distandardisasi, perbandingan besar koefisien menjadi lebih masuk akal.

Model ini masih punya beberapa keterbatasan. Linear Regression menganggap hubungan antara fitur dan harga rumah cukup lurus, padahal harga rumah bisa dipengaruhi banyak hal yang tidak ada di dataset. Model juga bisa terpengaruh outlier dan hubungan antar fitur yang saling berkaitan. Jadi, hasil prediksi sebaiknya dipakai sebagai perkiraan, bukan harga pasti.